In [2]:
# Import required Library
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error, root_mean_squared_error
import numpy as np

In [3]:
# Database setting
from sqlalchemy import create_engine
import pymysql

username = "root"
password = ""
port = 3306
database = "common"

engine = create_engine('mysql+pymysql://%s@localhost:%i/%s' %(username, port, database))

In [4]:
def get_evaluation(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    # mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    # median_ae = median_absolute_error(y_test, y_pred)

    print(f'Mean Absolute Error: {mae:.2f}')
    print(f'Mean Squared Error: {mse:.2f}')
    print(f'Root Mean Squared Error: {rmse:.2f}')
    print(f'R2 Score: {r2:.2f}')

    # return [mae, mse, rmse, r2, mape, median_ae]

In [5]:
def save_prediction(y_test, y_predict, file_name):
    y_predict_flat = y_predict.flatten()
    y_predict_series = pd.Series(y_predict_flat, name='Predicted')

    y_test.reset_index(drop=True, inplace=True)
    y_predict_series.reset_index(drop=True, inplace=True)

    combined_df = pd.concat([y_test, y_predict_series], axis=1)
    combined_df.to_csv(file_name, index=False)

# Load the Dataset from Database

## Load the Dataset

Only load the 'chr1' data.

In [6]:
sql = "SELECT H3K4me3, H3K9ac, H3K9me3, H3K27ac, H3K27me3, histone_total_count, h_value_1, h_value_2 \
       FROM common.histone_count_overlap80 \
       WHERE h_chrom = 'chr1'"
histone_count_df = pd.read_sql_query(sql, engine)

display(histone_count_df.head())
display(histone_count_df.shape)

,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3,histone_total_count,h_value_1,h_value_2
0,0,0,0,0,0,0,0.000000,0.000000
1,0,0,0,0,0,0,0.047392,0.034159
2,0,0,0,0,0,0,0.000000,0.000000
3,0,0,0,0,0,0,2.468570,2.805800
4,0,0,0,0,0,0,2.468570,2.805800


(5496, 8)

In [7]:
histone_count_df.describe()

,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3,histone_total_count,h_value_1,h_value_2
count,5496.000000,5496.000000,5496.000000,5496.000000,5496.000000,5496.000000,5496.000000,5496.000000
mean,3.903930,3.141376,0.044942,2.631004,0.142467,9.863719,21.836827,14.475264
std,3.448286,3.103233,0.253779,2.875129,0.586076,8.926012,228.272993,102.108659
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.017645,0.023875
50%,4.000000,3.000000,0.000000,2.000000,0.000000,9.000000,2.139950,2.098120
75%,7.000000,5.000000,0.000000,5.000000,0.000000,16.000000,11.243450,11.093200
max,16.000000,14.000000,3.000000,19.000000,8.000000,48.000000,5475.900000,5322.990000


In [56]:
## Whole Data

sql = "SELECT H3K4me3, H3K9ac, H3K9me3, H3K27ac, H3K27me3, histone_total_count, h_value_1, h_value_2 \
       FROM common.histone_count_overlap80"

all_df = pd.read_sql_query(sql, engine)

display(all_df.head())
display(all_df.shape)

,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3,histone_total_count,h_value_1,h_value_2
0,0,0,0,0,0,0,0.000000,0.000000
1,0,0,0,0,0,0,0.047392,0.034159
2,0,0,0,0,0,0,0.000000,0.000000
3,0,0,0,0,0,0,2.468570,2.805800
4,0,0,0,0,0,0,2.468570,2.805800


(59366, 8)

## Split into train test

### 'chr1'

In [8]:
X = histone_count_df[['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']]
y = histone_count_df[['h_value_1']]

display(X.head())
display(y.head())

,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3
0,0,0,0,0,0
1,0,0,0,0,0
2,0,0,0,0,0
3,0,0,0,0,0
4,0,0,0,0,0


,h_value_1
0,0.000000
1,0.047392
2,0.000000
3,2.468570
4,2.468570


In [9]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

display(X_train)
display(y_train)
display(X_test)
display(y_test)

,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3
4300,0,0,0,0,0
1812,0,0,1,0,0
2443,0,0,0,0,1
3855,3,3,0,2,0
5330,3,0,0,0,0
...,...,...,...,...,...
5218,0,0,0,0,0
4060,7,4,0,4,0
1346,0,0,0,0,0
3454,8,7,0,3,0


,h_value_1
4300,0.000000
1812,0.000000
2443,0.024852
3855,6.649030
5330,0.186278
...,...
5218,0.881686
4060,5.649670
1346,0.334225
3454,8.747720


,H3K4me3,H3K9ac,H3K9me3,H3K27ac,H3K27me3
4505,0,0,0,0,1
5462,8,8,0,6,0
2658,7,7,0,5,0
1448,5,4,0,3,0
4701,0,0,1,0,2
...,...,...,...,...,...
2499,4,4,0,5,0
2652,4,2,0,2,0
640,8,7,0,9,0
4699,0,0,1,0,2


,h_value_1
4505,0.000000
5462,0.000000
2658,25.274300
1448,41.478600
4701,16.643000
...,...
2499,8.513780
2652,248.611000
640,5.644510
4699,0.000000


### Whole Data

In [57]:
X_all = all_df[['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']]
y_all = all_df[['h_value_1']]

In [60]:
X_all_train, X_all_test, y_all_train, y_all_test = train_test_split(X_all, y_all, test_size=0.2, random_state=123)

display(X_all_train.shape)
display(X_all_test.shape)

(47492, 5)

(11874, 5)

# Modeling

## Random Forest Regressor

In [61]:
# Initialize the RandomForestRegressor model
rf = RandomForestRegressor()

# Set the parameters for grid search
param_grid = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', verbose=2, n_jobs=-1)

In [51]:
# Perform the grid search
grid_search.fit(X_train, y_train.values.ravel())

# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f'Best parameters: {best_params}')

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best parameters: {'bootstrap': True, 'max_depth': 20, 'min_samples_leaf': 4, 'min_samples_split': 5, 'n_estimators': 10}


In [52]:
# Make predictions on the test data using the best model
y_pred = best_model.predict(X_test)

In [53]:
# Make evaluation from the prediction
get_evaluation(y_test, y_pred)

Mean Absolute Error: 33.74
Mean Squared Error: 85514.96
Root Mean Squared Error: 292.43
R2 Score: -0.02


In [54]:
save_prediction(y_test, y_pred, "predictions/predict_random_forest_regressor.csv")

### GridSearch for All Data

In [66]:
# Initialize the RandomForestRegressor model
rf2 = RandomForestRegressor(bootstrap=True, max_depth=20, min_samples_leaf=4, min_samples_split = 5, n_estimators = 10)

In [67]:
rf2.fit(X_all_train, y_all_train)

d:\anaconda3\envs\epi-thesis\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(max_depth=20, min_samples_leaf=4, min_samples_split=5,
                      n_estimators=10)

In [ ]:
y_all_pred = rf2.predict(y_all_test)

In [ ]:
# Perform the grid search
grid_search.fit(X_all_train, y_all_train.values.ravel())

# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f'Best parameters: {best_params}')

## Support Vector Regressor

In [34]:
# Create an SVR Model

svr = SVR()

# Set the parameters for grid search
param_grid = {
    # 'kernel': ['linear', 'rbf', 'poly'],
    # 'kernel': ['linear', 'rbf'],
    # 'C': [0.1, 1, 10, 100],
    # 'gamma': [0.01, 0.1, 1, 10],
    # 'epsilon': [0.01, 0.1, 1]
    # 'degree': [2, 3, 4], # for poly kernel
    
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10],
    'gamma': [0.01, 0.1, 1],
    'epsilon': [0.01, 0.1, 1]
}

In [35]:
# Perform the GridSearch

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=svr, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', verbose=2)

# Perform the grid search
grid_search.fit(X_train, y_train.values.ravel())

# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f'Best parameters: {best_params}')

Fitting 5 folds for each of 54 candidates, totalling 270 fits
[CV] END .....C=0.1, epsilon=0.01, gamma=0.01, kernel=linear; total time=   0.2s
[CV] END .....C=0.1, epsilon=0.01, gamma=0.01, kernel=linear; total time=   0.2s
[CV] END .....C=0.1, epsilon=0.01, gamma=0.01, kernel=linear; total time=   0.2s
[CV] END .....C=0.1, epsilon=0.01, gamma=0.01, kernel=linear; total time=   0.2s
[CV] END .....C=0.1, epsilon=0.01, gamma=0.01, kernel=linear; total time=   0.2s
[CV] END ........C=0.1, epsilon=0.01, gamma=0.01, kernel=rbf; total time=   0.3s
[CV] END ........C=0.1, epsilon=0.01, gamma=0.01, kernel=rbf; total time=   0.3s
[CV] END ........C=0.1, epsilon=0.01, gamma=0.01, kernel=rbf; total time=   0.3s
[CV] END ........C=0.1, epsilon=0.01, gamma=0.01, kernel=rbf; total time=   0.3s
[CV] END ........C=0.1, epsilon=0.01, gamma=0.01, kernel=rbf; total time=   0.3s
[CV] END ......C=0.1, epsilon=0.01, gamma=0.1, kernel=linear; total time=   0.2s
[CV] END ......C=0.1, epsilon=0.01, gamma=0.1, 

In [36]:
# Make predictions on the test data using the best model
y_pred = best_model.predict(X_test)

In [37]:
# Make evaluation from the prediction
get_evaluation(y_test, y_pred)

Mean Absolute Error: 26.19
Mean Squared Error: 84456.13
Root Mean Squared Error: 290.61
R2 Score: -0.01


In [38]:
save_prediction(y_test, y_pred, "predictions/predict_support_vector_regressor.csv")

## XGBoost Regression

In [41]:
import xgboost as xgb
from xgboost import XGBRegressor

In [42]:
# Create an XGBoost model

xg_reg = XGBRegressor()

In [43]:
# Set the parameters for grid search
param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [44]:
# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=xg_reg, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', verbose=2, n_jobs=-1)

# Perform the grid search
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, gamma=None,
                                    grow_policy=None, importance_type=None,
                                    interaction_constraints=None,
                                    learning_rate=None, m...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None,
                                    random_state=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='neg_mean_squared_error', verbose=2)

In [45]:
# Get the best parameters and the best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

print(f'Best parameters: {best_params}')

Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}


In [46]:
# Make predictions on the test data using the best model
y_pred = best_model.predict(X_test)

In [47]:
# Make evaluation from the prediction
get_evaluation(y_test, y_pred)

Mean Absolute Error: 33.98
Mean Squared Error: 83664.26
Root Mean Squared Error: 289.25
R2 Score: 0.00


In [48]:
save_prediction(y_test, y_pred, "predictions/predict_xgboost.csv")